<div class="blog-language-switch" role="group" aria-label="Article language">
<span aria-current="page">English</span>
<a href="../zh-CN/Machine-Learning/07-linear-generalized-linear-models.html" lang="zh-CN" hreflang="zh-CN">中文</a>
</div>

[Back to Machine Learning guideline](Machine Learning.html)


## **Linear Regression and Generalized Linear Models**

Linear models are the most transparent place to connect representation, probability, optimization, regularization, and interpretation. Given a transformed feature vector $\phi(x)$, they begin with one additive score,

$$
\eta(x)=\beta_0+\phi(x)^\top\beta.
$$

The word **linear** refers to linearity in the fitted coefficients $\beta$. The response need not be a straight line in the original input: polynomial terms, interactions, and spline bases can enter $\phi(x)$, while a nonlinear inverse link can map $\eta$ to a probability or positive rate.

![Observed features may be expanded and combined into one linear predictor, then mapped through an outcome-specific inverse link.](assets/linear-model-family.svg){fig-align="center" width="100%" fig-alt="Feature map and linear predictor branching to Gaussian, Bernoulli, and Poisson response models"}

This chapter uses linear models in two complementary ways:

- as **predictors**, judged on unseen data using the protocols from Chapter 06;
- as **statistical models**, whose coefficients and uncertainty have valid interpretations only under explicit sampling and distributional assumptions.

Good predictive accuracy does not make a coefficient causal. A coefficient describes a conditional association under the specified features, link, data-generating population, and regularization. Causal interpretation additionally requires an identification argument about treatment assignment, confounding, selection, measurement, and interference.


### **Linear Predictive Models**

Linear regression and logistic regression share the same systematic component $\eta=\beta_0+x^\top\beta$, but they differ in the response support and fitting likelihood. Linear regression maps $\eta$ directly to a real-valued conditional mean. Logistic regression maps it through the sigmoid to a probability in $[0,1]$.

#### **Linear Regression**

Linear regression models a conditional mean rather than claiming that every observed point lies on a line:

$$
\mathbb E[Y\mid X=x]=\beta_0+x^\top\beta.
$$

An observation can be written as $Y_i=\beta_0+x_i^\top\beta+\varepsilon_i$, where $\varepsilon_i$ represents variation not explained by the modeled features. The model can support prediction under relatively weak conditions; classical confidence intervals and tests require stronger assumptions discussed later.

##### **Simple (Bivariate) Linear Regression**

With one predictor,

$$
\widehat y=\widehat\beta_0+\widehat\beta_1x.
$$

$\widehat\beta_1$ is the estimated change in the conditional mean of $Y$ for a one-unit increase in $x$; $\widehat\beta_0$ is the fitted mean at $x=0$. The intercept may be mathematically necessary but scientifically meaningless when zero lies far outside the observed range. The fitted slope summarizes association over the modeled range and should not be extrapolated through regimes unsupported by data.

For ordinary least squares with an intercept, the closed-form simple-regression estimates are

$$
\widehat\beta_1=
\frac{\sum_i(x_i-\overline x)(y_i-\overline y)}
{\sum_i(x_i-\overline x)^2},
\qquad
\widehat\beta_0=\overline y-\widehat\beta_1\overline x.
$$

The slope is covariance divided by predictor variance. Correlation instead standardizes both variables, so it is unitless and symmetric; a regression slope has units and assigns distinct response and predictor roles.

![Ordinary least squares chooses the line that minimizes the sum of squared vertical residuals.](assets/ols-fit.png){fig-align="center" width="68%" fig-alt="Observed points and a fitted ordinary least-squares regression line"}

*Image: [scikit-learn, Linear Regression Example](https://scikit-learn.org/0.24/auto_examples/linear_model/plot_ols.html).*

<details>
<summary><strong>Python example: compute the simple-regression slope by hand and verify it</strong></summary>

```python
import numpy as np
from sklearn.linear_model import LinearRegression

x = np.array([1.0, 2.0, 3.0, 5.0, 7.0, 8.0])
y = np.array([2.1, 3.9, 5.8, 9.2, 12.7, 15.1])

# Closed-form ordinary least-squares estimates with an intercept.
slope = np.sum((x - x.mean()) * (y - y.mean())) / np.sum((x - x.mean()) ** 2)
intercept = y.mean() - slope * x.mean()

model = LinearRegression().fit(x.reshape(-1, 1), y)

print("manual intercept and slope:", np.round([intercept, slope], 4))
print("sklearn intercept and slope:", np.round([model.intercept_, model.coef_[0]], 4))
print("prediction at x=6:", round(float(model.predict([[6.0]])[0]), 3))
print("training residuals:", np.round(y - model.predict(x.reshape(-1, 1)), 3))
```

</details>

##### **Multiple Linear Regression**

With $p$ predictors,

$$
\mathbb E[Y\mid X=x]
=\beta_0+\beta_1x_1+\cdots+\beta_px_p.
$$

$\beta_j$ is a **partial association**: the expected change in $Y$ for a one-unit increase in $x_j$ while the other modeled predictors are held fixed. This interpretation can be unrealistic when predictors cannot vary independently, when important interactions are absent, or when a one-unit intervention would change other variables.

Categorical predictors enter through indicator or contrast columns. An interaction $x_1x_2$ permits the slope of $x_1$ to depend on $x_2$:

$$
\frac{\partial\mathbb E[Y\mid X]}{\partial x_1}
=\beta_1+\beta_{12}x_2.
$$

Main effects should usually remain when their interaction is included, because deleting them imposes the strong constraint that each variable has zero effect whenever the other equals zero.

<details>
<summary><strong>Python example: multiple regression separates effects hidden by correlated predictors</strong></summary>

```python
import numpy as np
from sklearn.linear_model import LinearRegression

rng = np.random.default_rng(113)
n = 1200
x1 = rng.normal(size=n)
x2 = 0.85 * x1 + rng.normal(scale=0.45, size=n)  # strongly correlated with x1
y = 3.0 * x1 - 2.0 * x2 + rng.normal(scale=0.5, size=n)

simple = LinearRegression().fit(x1[:, None], y)
multiple = LinearRegression().fit(np.column_stack([x1, x2]), y)

print("true coefficients:                 ", [3.0, -2.0])
print("x1 coefficient when x2 is omitted:", round(simple.coef_[0], 3))
print("multiple-regression coefficients:  ", np.round(multiple.coef_, 3))
print("correlation between x1 and x2:     ", round(np.corrcoef(x1, x2)[0, 1], 3))
```

</details>

The simple slope absorbs part of the omitted predictor's association because $x_1$ and $x_2$ move together. Including a variable does not automatically “control for confounding” in a causal sense; the feature set and functional form must follow a defensible causal structure.

##### **Model Fitting**

After adding a column of ones for the intercept, write the design matrix as $X\in\mathbb R^{n\times(p+1)}$, response as $y\in\mathbb R^n$, and coefficients as $\beta$. Ordinary least squares solves

$$
\widehat\beta
=\arg\min_\beta\|y-X\beta\|_2^2.
$$

Setting the gradient to zero gives the normal equations

$$
X^\top X\widehat\beta=X^\top y.
$$

If $X$ has full column rank, $\widehat\beta=(X^\top X)^{-1}X^\top y$. Software generally uses QR decomposition, singular-value decomposition, or iterative solvers rather than explicitly forming the inverse, because $X^\top X$ squares the condition number and can magnify numerical error.

![Ordinary least squares is the orthogonal projection of the observed response onto the column space of the design matrix.](assets/ols-projection-geometry.svg){fig-align="center" width="100%" fig-alt="Response vector projected onto the column space of X with an orthogonal residual"}

At the optimum, the residual $e=y-X\widehat\beta$ satisfies $X^\top e=0$: it is orthogonal to every design column. The fitted vector $\widehat y$ can remain unique even when coefficients are not. If columns are linearly dependent, infinitely many coefficient vectors produce the same fitted values; a pseudoinverse chooses one minimum-norm solution.

Under independent Gaussian errors $\varepsilon_i\sim\mathcal N(0,\sigma^2)$, minimizing squared error is equivalent to maximizing likelihood. Under zero conditional mean and constant variance, the Gauss-Markov theorem makes OLS the best linear unbiased estimator, but “best” means minimum variance within the class of linear unbiased estimators, not best predictor among all possible nonlinear methods.

<details>
<summary><strong>Python example: rank deficiency destabilizes coefficients more than predictions</strong></summary>

```python
import numpy as np
from sklearn.linear_model import LinearRegression

rng = np.random.default_rng(127)
x1 = rng.normal(size=300)
x2 = rng.normal(size=300)
x3 = x1 + x2  # exact linear dependence
X = np.column_stack([x1, x2, x3])
y = 2.0 * x1 - 1.0 * x2 + rng.normal(scale=0.05, size=300)

base = LinearRegression().fit(X, y)

# A tiny perturbation breaks the exact dependence and can change the coefficient allocation.
X_perturbed = X.copy()
X_perturbed[:, 2] += rng.normal(scale=1e-5, size=len(X))
perturbed = LinearRegression().fit(X_perturbed, y)

prediction_change = np.max(np.abs(base.predict(X) - perturbed.predict(X_perturbed)))

print("design rank / number of columns:", np.linalg.matrix_rank(X), X.shape[1])
print("coefficients on rank-deficient X:", np.round(base.coef_, 3))
print("coefficients after tiny perturbation:", np.round(perturbed.coef_, 3))
print("maximum fitted-value change:", f"{prediction_change:.6f}")
```

</details>

##### **Evaluation Metrics**

Chapter 06 defined MAE, RMSE, $R^2$, residual slicing, uncertainty, and valid data splits. For linear models, three additions matter.

First, $R^2$ compares squared error with an intercept-only mean baseline:

$$
R^2=1-\frac{\sum_i(y_i-\widehat y_i)^2}{\sum_i(y_i-\overline y)^2}.
$$

It can be negative on unseen data and does not certify correct functional form. Adjusted $R^2$ penalizes the number of fitted slopes,

$$
\overline R^2
=1-(1-R^2)\frac{n-1}{n-p-1},
$$

but remains an in-sample criterion and is not a substitute for validation.

Second, uncertainty about the **conditional mean** differs from uncertainty about a **new observation**. A confidence interval for $\mathbb E[Y\mid X=x_*]$ reflects coefficient uncertainty. A prediction interval also includes irreducible observation noise and is therefore wider.

Third, conventional standard errors assume the model and sampling process used to derive them. Cross-validation measures predictive stability under resampling; it does not automatically provide valid coefficient inference, while a small coefficient $p$-value does not establish useful out-of-sample prediction.

<details>
<summary><strong>Python example: compare a mean-response interval with a prediction interval</strong></summary>

```python
import numpy as np
from scipy.stats import t

rng = np.random.default_rng(131)
x = rng.uniform(0, 10, size=180)
y = 4.0 + 1.7 * x + rng.normal(scale=2.2, size=len(x))

X = np.column_stack([np.ones(len(x)), x])
beta = np.linalg.lstsq(X, y, rcond=None)[0]
residual = y - X @ beta
n, p = X.shape
residual_variance = residual @ residual / (n - p)
xtx_inverse = np.linalg.inv(X.T @ X)

new_row = np.array([1.0, 6.0])
mean_prediction = new_row @ beta
mean_variance = residual_variance * (new_row @ xtx_inverse @ new_row)

# A new observation contains both mean-estimation uncertainty and fresh noise.
mean_se = np.sqrt(mean_variance)
observation_se = np.sqrt(mean_variance + residual_variance)
critical_value = t.ppf(0.975, df=n - p)
mean_interval = mean_prediction + critical_value * np.array([-mean_se, mean_se])
observation_interval = mean_prediction + critical_value * np.array([
    -observation_se, observation_se
])

print("estimated coefficients:", np.round(beta, 3))
print("fitted mean at x=6:", round(mean_prediction, 3))
print("95% mean-response interval:", np.round(mean_interval, 3))
print("95% new-observation interval:", np.round(observation_interval, 3))
print("The observation interval is wider because it includes new-case noise.")
```

</details>

#### **Logistic Regression**

Logistic regression models a Bernoulli probability while retaining a linear predictor. For $Y\in\{0,1\}$,

$$
p(x)=\Pr(Y=1\mid X=x)
=\sigma(\eta)
=\frac{1}{1+e^{-\eta}},
\qquad
\eta=\beta_0+x^\top\beta.
$$

Applying the logit link gives

$$
\log\frac{p(x)}{1-p(x)}=\beta_0+x^\top\beta.
$$

Thus $\beta_j$ is an additive change in **log odds**, and $e^{\beta_j}$ is the multiplicative change in odds for one unit of $x_j$, holding modeled features fixed. An odds ratio is not a risk ratio: when the event is common, they can differ substantially.

Maximum likelihood minimizes binary cross-entropy,

$$
-\sum_i\left[y_i\log p_i+(1-y_i)\log(1-p_i)\right],
$$

which has no ordinary closed-form coefficient solution. Newton, quasi-Newton, coordinate-descent, or stochastic methods are used. The probability surface is nonlinear, but the $p=0.5$ boundary is the hyperplane $\eta=0$. A deployment threshold other than $0.5$ moves that parallel boundary without refitting the probability model.

<details>
<summary><strong>Python example: reconstruct logistic probabilities and interpret standardized odds ratios</strong></summary>

```python
import numpy as np
from scipy.special import expit
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

data = load_breast_cancer()
X, y = data.data, data.target
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(C=1.0, max_iter=2000, random_state=137),
).fit(X, y)

# Choose an observed case near the fitted boundary so both outcomes remain plausible.
all_probabilities = model.predict_proba(X)[:, 1]
row_index = np.argmin(np.abs(all_probabilities - 0.5))
scaled_row = model[0].transform(X[[row_index]])
linear_score = model[1].intercept_[0] + scaled_row @ model[1].coef_[0]
manual_probability = expit(linear_score)[0]
api_probability = model.predict_proba(X[[row_index]])[0, 1]

largest = np.argsort(np.abs(model[1].coef_[0]))[-3:][::-1]
print("manual / API probability:", round(manual_probability, 6), round(api_probability, 6))
print("largest one-standard-deviation odds multipliers:")
for index in largest:
    print(f"  {data.feature_names[index]:28s} exp(beta)={np.exp(model[1].coef_[0, index]):.3f}")
```

</details>

When data are completely separable, an unregularized likelihood can keep increasing as coefficient magnitudes diverge. Predictions approach 0 or 1, but finite maximum-likelihood coefficients may not exist. Regularization produces a finite, more stable solution; it also changes coefficient interpretation from an unpenalized likelihood estimate to a deliberately biased predictive estimate.

<details>
<summary><strong>Python example: regularization controls coefficient growth under separation</strong></summary>

```python
import numpy as np
from sklearn.linear_model import LogisticRegression

x = np.r_[np.linspace(-3.0, -0.2, 40), np.linspace(0.2, 3.0, 40)][:, None]
y = np.r_[np.zeros(40, dtype=int), np.ones(40, dtype=int)]

models = {
    "almost unregularized": LogisticRegression(C=1e8, max_iter=5000),
    "regularized": LogisticRegression(C=1.0, max_iter=5000),
}

for name, model in models.items():
    model.fit(x, y)
    near_boundary = model.predict_proba([[-0.2], [0.2]])[:, 1]
    print(
        f"{name:20s}",
        "coefficient =", round(model.coef_[0, 0], 3),
        "P(y=1 | x=-0.2, 0.2) =", np.round(near_boundary, 4),
    )
```

</details>

Linear and logistic regression are effective when additive effects on the response or link scale are plausible, sample size supports the number of coefficients, and extrapolation is controlled. Basis expansions address smooth nonlinear effects; regularization addresses variance and ill-conditioning; the GLM framework makes the response distribution explicit.


### **The Generalized Linear Model Framework**

A **generalized linear model (GLM)** extends linear regression by separating three choices:

1. a conditional distribution appropriate for the response support and variance;
2. a linear predictor containing the systematic feature effects;
3. a link connecting the conditional mean to that predictor.

![A generalized linear model combines a random component, a linear predictor, and a link function.](assets/glm-framework.svg){fig-align="center" width="100%" fig-alt="Three components of a generalized linear model and common Gaussian, Bernoulli, and Poisson combinations"}

The framework models

$$
\mu_i=\mathbb E[Y_i\mid X_i],
\qquad
g(\mu_i)=\eta_i=\beta_0+x_i^\top\beta.
$$

The relationship is linear on the **link scale** $\eta$, not necessarily on the response scale $\mu$. Choosing a distribution and link is a claim about valid predictions, conditional variance, and coefficient meaning.

#### **Random Component**

The random component specifies $Y_i\mid X_i$ and therefore its support and mean-variance relationship. A GLM commonly writes

$$
\operatorname{Var}(Y_i\mid X_i)=\phi V(\mu_i),
$$

where $V(\mu)$ is a family-specific variance function and $\phi$ is a dispersion parameter when one is available.

| Family | Response support | Conditional variance | Typical use |
|---|---|---|---|
| Gaussian | real values | $\phi$ | approximately constant-variance continuous outcomes |
| Bernoulli | $0$ or $1$ | $\mu(1-\mu)$ | binary events |
| Binomial | $0,\ldots,m_i$ | $m_i\pi_i(1-\pi_i)$ | successes among known trials |
| Poisson | nonnegative integers | $\mu$ | event counts or rates with exposure |
| Gamma | positive continuous | $\phi\mu^2$ | skewed positive costs or durations |

The distribution is conditional, not a statement that the marginal histogram of $Y$ must have the same shape. A mixture of Gaussian conditional distributions can have a non-Gaussian marginal distribution because their means vary with $X$.

Poisson equality of mean and variance is often too restrictive. **Overdispersion** occurs when conditional variance exceeds the assumed Poisson variance, commonly because of unmodeled heterogeneity, dependence, or excess zeros. Standard errors and prediction intervals can then be too narrow even if the conditional mean is useful. Negative-binomial models or quasi-likelihood variance corrections may be more appropriate.

<details>
<summary><strong>Python example: compare empirical mean-variance relationships across response families</strong></summary>

```python
import numpy as np

rng = np.random.default_rng(139)
means = [0.2, 0.5, 0.8]

print("Bernoulli samples")
for mean in means:
    sample = rng.binomial(1, mean, size=100_000)
    print(f"  mean={sample.mean():.3f}, variance={sample.var():.3f}, theory={mean * (1 - mean):.3f}")

print("\nPoisson samples")
for mean in [0.5, 2.0, 8.0]:
    sample = rng.poisson(mean, size=100_000)
    print(f"  mean={sample.mean():.3f}, variance={sample.var():.3f}, theory={mean:.3f}")

print("\nGaussian samples with constant variance")
for mean in [-2.0, 0.0, 3.0]:
    sample = rng.normal(mean, 1.5, size=100_000)
    print(f"  mean={sample.mean():.3f}, variance={sample.var():.3f}, theory={1.5 ** 2:.3f}")
```

</details>

#### **Linear Predictor**

The systematic component is

$$
\eta_i=\beta_0+x_i^\top\beta+o_i,
$$

where $o_i$ is an optional **offset** whose coefficient is fixed at one. The predictor can contain continuous variables, encoded categories, interactions, polynomial or spline bases, and known offsets while remaining linear in $\beta$.

Offsets are essential when observations have unequal opportunity for an event. If $Y_i$ is a count observed over exposure $E_i$, a Poisson rate model uses

$$
\log\mu_i
=\log E_i+\beta_0+x_i^\top\beta,
$$

so

$$
\frac{\mu_i}{E_i}=\exp(\beta_0+x_i^\top\beta).
$$

The known $\log E_i$ offset scales expected count proportionally with exposure. Treating exposure as an ordinary learned feature would estimate a coefficient that should be fixed by the measurement process, while omitting exposure confuses opportunity with risk.

<details>
<summary><strong>Python example: recover a Poisson event-rate effect with an exposure offset</strong></summary>

```python
import numpy as np
from sklearn.linear_model import PoissonRegressor

rng = np.random.default_rng(149)
n = 2500
risk_feature = rng.normal(size=n)

# Exposure is associated with risk here, making omission visibly misleading.
log_exposure = 0.5 * risk_feature + rng.normal(scale=0.45, size=n)
exposure = np.clip(np.exp(log_exposure), 0.1, 4.0)

# The true event rate per exposure unit is exp(-1.2 + 0.65 * risk_feature).
true_rate = np.exp(-1.2 + 0.65 * risk_feature)
count = rng.poisson(exposure * true_rate)
X = risk_feature[:, None]

# Fitting count / exposure with exposure as sample weight is likelihood-
# equivalent to a Poisson count model with log(exposure) as a fixed offset.
with_offset = PoissonRegressor(alpha=0.0, max_iter=1_000).fit(
    X,
    count / exposure,
    sample_weight=exposure,
)
without_offset = PoissonRegressor(alpha=0.0, max_iter=1_000).fit(X, count)

print("true intercept and slope:       ", [-1.2, 0.65])
print("with exposure adjustment:       ", np.round([
    with_offset.intercept_, with_offset.coef_[0]
], 3))
print("without exposure adjustment:    ", np.round([
    without_offset.intercept_, without_offset.coef_[0]
], 3))
print("rate ratio for +1 feature unit: ", round(np.exp(with_offset.coef_[0]), 3))
```

</details>

The rate ratio $e^{\beta_j}$ is multiplicative on the expected rate, holding other modeled variables and exposure fixed. As with odds ratios, it is an association unless a causal design supports a causal interpretation.

#### **Link Function**

The link $g$ maps the valid mean domain to the unrestricted real line used by the predictor:

$$
g(\mu)=\eta,
\qquad
\mu=g^{-1}(\eta).
$$

An identity inverse link permits any real mean. A sigmoid constrains a Bernoulli mean to $[0,1]$. An exponential constrains a Poisson or Gamma mean to be positive. The link also determines coefficient interpretation:

- identity: $\beta_j$ is an additive mean difference per feature unit;
- logit: $e^{\beta_j}$ is an odds multiplier;
- log: $e^{\beta_j}$ is a mean or rate multiplier.

A **canonical link** connects the natural parameter of an exponential-family distribution directly to $\eta$ and often simplifies likelihood equations. Gaussian identity, Bernoulli logit, and Poisson log are canonical combinations. Canonical does not mean universally correct: scientific interpretation, numerical behavior, and calibration may motivate another valid link.

The link concerns the conditional mean, not an arbitrary transformation of observed $y$. Fitting least squares to $\log(y)$ and exponentiating is generally not equivalent to a log-link GLM because $\mathbb E[e^{\log Y}\mid X]$ and $e^{\mathbb E[\log Y\mid X]}$ differ, and zeros cannot be logged without an ad hoc adjustment.

<details>
<summary><strong>Python example: a Poisson log link guarantees nonnegative count predictions</strong></summary>

```python
import numpy as np
from sklearn.linear_model import LinearRegression, PoissonRegressor
from sklearn.metrics import mean_poisson_deviance
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(151)
X = rng.uniform(-3.5, 2.5, size=(3000, 1))
true_mean = np.exp(-0.8 + 0.75 * X[:, 0])
y = rng.poisson(true_mean)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.35, random_state=151
)

ols = LinearRegression().fit(X_train, y_train)
poisson = PoissonRegressor(alpha=0.0, max_iter=1000).fit(X_train, y_train)

ols_prediction = ols.predict(X_test)
poisson_prediction = poisson.predict(X_test)

print("minimum OLS prediction:    ", round(ols_prediction.min(), 3))
print("minimum Poisson prediction:", round(poisson_prediction.min(), 3))
print("negative OLS predictions:  ", int(np.sum(ols_prediction <= 0)))
print("Poisson test deviance:      ", round(mean_poisson_deviance(y_test, poisson_prediction), 3))

# Poisson deviance is undefined for nonpositive predictions, so clipping OLS is only a diagnostic workaround.
clipped_ols = np.clip(ols_prediction, 1e-8, None)
print("clipped-OLS test deviance:  ", round(mean_poisson_deviance(y_test, clipped_ols), 3))
```

</details>

#### **Exponential-Family Distributions**

A one-parameter exponential-family density can be written as

$$
p(y\mid\theta,\phi)
=\exp\left(
\frac{y\theta-b(\theta)}{a(\phi)}+c(y,\phi)
\right),
$$

where $\theta$ is the natural parameter, $\phi$ controls dispersion, and $b$ determines moments:

$$
\mathbb E[Y]=b'(\theta),
\qquad
\operatorname{Var}(Y)=a(\phi)b''(\theta).
$$

This form unifies Gaussian, Bernoulli/binomial, Poisson, Gamma, and inverse-Gaussian families. It provides likelihoods, score equations, deviance residuals, and iterative reweighted least-squares fitting under one mathematical framework.

**Deviance** compares a fitted model with a saturated model that fits every observation as closely as the family permits:

$$
D=2\left[\ell(\text{saturated})-\ell(\text{fitted})\right].
$$

Lower deviance indicates better likelihood fit on the same data, but training deviance always tends to improve with flexibility. Out-of-sample deviance or a penalized information criterion is needed for model comparison. Deviance values from different response families are not generally interchangeable.

<details>
<summary><strong>Python example: compute Poisson deviance contributions and identify poorly fitted counts</strong></summary>

```python
import numpy as np

y = np.array([0, 1, 2, 8, 3, 12], dtype=float)
predicted_mean = np.array([0.4, 0.8, 2.5, 4.0, 3.2, 7.0])

# For y=0, the limit of y * log(y / mu) is zero.
log_term = np.zeros_like(y)
positive = y > 0
log_term[positive] = y[positive] * np.log(y[positive] / predicted_mean[positive])
contribution = 2 * (log_term - (y - predicted_mean))

for observed, predicted, deviance in zip(y, predicted_mean, contribution):
    print(f"observed={observed:>4.0f}, predicted={predicted:>4.1f}, deviance={deviance:.3f}")
print("total mean Poisson deviance:", round(contribution.mean(), 3))
```

</details>

**Framework comparison.** Gaussian identity regression is appropriate for real-valued outcomes with approximately constant conditional variance. Bernoulli-logit regression models event probabilities. Poisson-log regression models nonnegative counts or rates and can incorporate exposure through offsets. Gamma-log regression handles positive right-skewed means. Diagnostics must assess both the linear predictor and the chosen mean-variance relationship.


### **Basis Expansion and Nonlinear Effects**

A straight line is often too restrictive, but abandoning linear models is not the only response. A **basis expansion** transforms the original input into derived features,

$$
x \longmapsto \phi(x)=[\phi_1(x),\ldots,\phi_m(x)]^\top,
\qquad
\eta(x)=\beta_0+\sum_{j=1}^{m}\beta_j\phi_j(x).
$$

The fitted curve can be nonlinear in $x$ while remaining linear in the unknown coefficients. Consequently, familiar least-squares or GLM optimization still applies. The representation determines which shapes are available; regularization and validation determine how much flexibility the data can support.

#### **Polynomial Regression**

Polynomial regression uses powers of a variable as basis functions. A degree-$d$ model has

$$
\widehat y=\beta_0+\beta_1x+\beta_2x^2+\cdots+\beta_dx^d.
$$

It is a linear model because the coefficients enter additively, even though the curve is nonlinear in $x$. With several predictors, `PolynomialFeatures` can also produce cross-products such as $x_1x_2$, which encode interactions: the effect of one variable can then depend on another.

Degree controls the bias-flexibility trade-off. A line cannot represent curvature; a moderate polynomial can; a high-degree polynomial may oscillate between observations, react strongly to noise, and extrapolate catastrophically. Raw powers also become highly correlated and numerically ill-conditioned. Centering or scaling $x$, using orthogonal polynomial bases, and regularizing the coefficients improve stability, but the degree must still be selected inside cross-validation.

![Linear, polynomial, and spline models fit the same nonlinear signal with different flexibility and extrapolation behavior.](assets/polynomial-spline-fit.png){fig-align="center" width="88%" fig-alt="Comparison of linear regression, polynomial regression, and spline interpolation on a nonlinear signal"}

*Image: [scikit-learn, Polynomial and Spline Interpolation](https://scikit-learn.org/stable/auto_examples/linear_model/plot_polynomial_interpolation.html).*

<details>
<summary><strong>Python example: compare polynomial degree using train and test error</strong></summary>

```python
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures

rng = np.random.default_rng(12)
x_train = np.sort(rng.uniform(-2.5, 2.5, 45))[:, None]
x_test = np.linspace(-3.0, 3.0, 300)[:, None]

def signal(x):
    return np.sin(1.5 * x) + 0.15 * x

y_train = signal(x_train[:, 0]) + rng.normal(0, 0.22, len(x_train))
y_test = signal(x_test[:, 0])

for degree in (1, 3, 10):
    model = make_pipeline(
        PolynomialFeatures(degree=degree, include_bias=False),
        LinearRegression(),
    )
    model.fit(x_train, y_train)
    train_mse = mean_squared_error(y_train, model.predict(x_train))
    test_mse = mean_squared_error(y_test, model.predict(x_test))
    edge_prediction = model.predict(np.array([[4.0]]))[0]
    print(
        f"degree={degree:>2}: train MSE={train_mse:.3f}, "
        f"test MSE={test_mse:.3f}, prediction at x=4={edge_prediction:.2f}"
    )
```

</details>

The degree-10 model can achieve the lowest training error while behaving poorly near or beyond the data boundary. This is why training fit is not a valid model-selection criterion.

<details>
<summary><strong>Python example: tune a regularized polynomial pipeline without preprocessing leakage</strong></summary>

```python
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

rng = np.random.default_rng(8)
X = rng.uniform(-3, 3, size=(120, 1))
y = 0.6 * X[:, 0] ** 2 - 0.25 * X[:, 0] ** 3 + rng.normal(0, 1.2, 120)

# Every fold learns its polynomial expansion and scaling from that fold's
# training partition. The test fold never influences these transformations.
pipeline = Pipeline([
    ("basis", PolynomialFeatures(include_bias=False)),
    ("scale", StandardScaler()),
    ("ridge", Ridge()),
])

search = GridSearchCV(
    pipeline,
    param_grid={
        "basis__degree": [1, 2, 3, 5, 8],
        "ridge__alpha": [0.01, 0.1, 1.0, 10.0],
    },
    scoring="neg_mean_squared_error",
    cv=KFold(n_splits=5, shuffle=True, random_state=8),
)
search.fit(X, y)

print("selected parameters:", search.best_params_)
print("cross-validated RMSE:", round(np.sqrt(-search.best_score_), 3))
```

</details>

#### **Splines and Generalized Additive Models**

A spline represents a smooth function by joining low-degree polynomial pieces at **knots**. B-spline basis functions have local support: changing one coefficient primarily changes a limited region rather than the entire curve. This local behavior usually makes splines more stable than a single high-degree global polynomial.

For a cubic spline, adjacent pieces are constrained to meet with continuous first and second derivatives. Flexibility depends on knot number and placement, polynomial degree, and any smoothness penalty. Too few basis functions underfit; too many unpenalized basis functions can reproduce noise.

![Polynomial powers affect the full input range, whereas B-spline basis functions are localized around neighboring knots.](assets/polynomial-spline-bases.png){fig-align="center" width="88%" fig-alt="Polynomial basis functions compared with localized B-spline basis functions"}

*Image: [scikit-learn, Polynomial and Spline Interpolation](https://scikit-learn.org/stable/auto_examples/linear_model/plot_polynomial_interpolation.html).*

A **generalized additive model (GAM)** combines smooth one-dimensional effects on the scale of a GLM linear predictor:

$$
g\!\left(\mathbb E[Y\mid X]\right)
=\beta_0+f_1(x_1)+f_2(x_2)+\cdots+f_p(x_p).
$$

Each $f_j$ is usually expressed with a spline basis and estimated with a roughness penalty. A GAM can reveal nonlinear partial effects while retaining more interpretability than an unrestricted black-box model. Its default additive form is also a limitation: interactions such as $f(x_1,x_2)$ must be included deliberately, and smooth curves remain associational rather than causal.

<details>
<summary><strong>Python example: compare a global polynomial with a locally supported spline</strong></summary>

```python
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, SplineTransformer, StandardScaler

rng = np.random.default_rng(31)
X_train = np.sort(rng.uniform(0, 8, 70))[:, None]
X_test = np.linspace(0, 8, 400)[:, None]

def signal(x):
    return np.sin(x) + 0.12 * x

y_train = signal(X_train[:, 0]) + rng.normal(0, 0.18, len(X_train))
y_test = signal(X_test[:, 0])

polynomial = make_pipeline(
    PolynomialFeatures(degree=9, include_bias=False),
    StandardScaler(),
    Ridge(alpha=0.1),
)
spline = make_pipeline(
    SplineTransformer(n_knots=7, degree=3, include_bias=False),
    Ridge(alpha=0.1),
)

for name, model in [("degree-9 polynomial", polynomial), ("cubic spline", spline)]:
    model.fit(X_train, y_train)
    interpolation_mse = mean_squared_error(y_test, model.predict(X_test))
    outside = model.predict(np.array([[-1.0], [9.0]]))
    print(
        f"{name:>20}: interpolation MSE={interpolation_mse:.4f}, "
        f"outside predictions={np.round(outside, 2)}"
    )
```

</details>

<details>
<summary><strong>Python example: inspect the local support of a B-spline basis</strong></summary>

```python
import numpy as np
from sklearn.preprocessing import SplineTransformer

X = np.linspace(0, 10, 11)[:, None]
basis = SplineTransformer(
    n_knots=6,
    degree=3,
    include_bias=False,
).fit_transform(X)

# A row contains the basis activations used to predict at one x value.
# Only a small neighborhood of cubic B-spline functions is active.
for index in (0, 5, 10):
    active = np.flatnonzero(np.abs(basis[index]) > 1e-10)
    print(
        f"x={X[index, 0]:>4.1f}: active basis columns={active.tolist()}, "
        f"values={np.round(basis[index, active], 3)}"
    )
```

</details>

**Method comparison.** Polynomial regression is compact and useful when a low-degree global trend is scientifically plausible. Splines provide local flexibility and usually safer interpolation. GAMs extend this idea to several predictors and non-Gaussian outcomes, but their smoothness and interaction structure must be validated. None of these methods makes unconstrained extrapolation reliable; outside the observed feature range, predictions are driven mainly by the chosen functional form.


### **Regularized Linear Models**

When predictors are numerous, correlated, or expressed through a large basis, minimizing training loss alone can produce unstable coefficients. **Regularization** adds a penalty that prefers simpler parameter vectors:

$$
\widehat\beta
=\arg\min_\beta\left[
\mathcal L(\beta)+\lambda\,\Omega(\beta)
\right].
$$

$\mathcal L$ measures data mismatch, $\Omega$ measures coefficient complexity, and $\lambda\ge 0$ controls the trade-off. A larger $\lambda$ increases bias but can reduce variance enough to improve unseen-data performance. The intercept is normally excluded from the penalty.

![L2 and L1 constraints meet elliptical loss contours differently: ridge shrinks continuously, whereas lasso often touches an axis and sets a coefficient to zero.](assets/regularization-geometry.svg){fig-align="center" width="88%" fig-alt="Geometric comparison of circular ridge and diamond-shaped lasso constraints against loss contours"}

Because penalties depend on coefficient magnitude, features must generally be placed on comparable scales. Scaling belongs inside the fitted pipeline so that each validation fold estimates its own preprocessing parameters. The penalty strength is a hyperparameter, not a value to choose by training error.

#### **Ridge Regression**

Ridge regression uses the squared $L_2$ norm:

$$
\widehat\beta_{\text{ridge}}
=\arg\min_\beta
\left\{
\lVert y-X\beta\rVert_2^2+\lambda\lVert\beta\rVert_2^2
\right\}.
$$

For centered data and an unpenalized intercept,

$$
\widehat\beta_{\text{ridge}}
=(X^\top X+\lambda I)^{-1}X^\top y.
$$

Adding $\lambda I$ stabilizes directions in which $X^\top X$ has very small eigenvalues. Ridge is therefore especially useful under multicollinearity: instead of allowing correlated predictors to receive large opposing coefficients, it spreads the signal and shrinks the group. Coefficients approach zero smoothly as regularization increases, but are rarely exactly zero, so ridge is not a feature-selection method.

![Ridge coefficient paths shrink toward zero as the regularization strength increases.](assets/ridge-coefficient-path.png){fig-align="center" width="78%" fig-alt="Ridge regression coefficient trajectories over increasing regularization strength"}

*Image: [scikit-learn, Ridge Coefficients as a Function of Regularization](https://scikit-learn.org/stable/auto_examples/linear_model/plot_ridge_coeffs.html).*

<details>
<summary><strong>Python example: show how ridge stabilizes coefficients under multicollinearity</strong></summary>

```python
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(22)
n = 180
latent = rng.normal(size=n)
X = np.column_stack([
    latent + rng.normal(scale=0.03, size=n),
    latent + rng.normal(scale=0.03, size=n),
    rng.normal(size=n),
])
y = 3.0 * latent + 0.5 * X[:, 2] + rng.normal(scale=0.7, size=n)

X_test = rng.normal(size=(300, 3))
# Recreate the same correlation structure in the test set.
test_latent = X_test[:, 0].copy()
X_test[:, 0] = test_latent + rng.normal(scale=0.03, size=300)
X_test[:, 1] = test_latent + rng.normal(scale=0.03, size=300)
y_test = 3.0 * test_latent + 0.5 * X_test[:, 2] + rng.normal(scale=0.7, size=300)

models = {
    "OLS": LinearRegression(),
    "Ridge": make_pipeline(StandardScaler(), Ridge(alpha=10.0)),
}
coefficient_samples = {name: [] for name in models}
test_errors = {name: [] for name in models}

for _ in range(120):
    rows = rng.integers(0, n, size=n)  # bootstrap resample
    for name, model in models.items():
        model.fit(X[rows], y[rows])
        fitted = model[-1] if name == "Ridge" else model
        coefficient_samples[name].append(fitted.coef_)
        test_errors[name].append(mean_squared_error(y_test, model.predict(X_test)))

for name in models:
    coefficient_sd = np.std(coefficient_samples[name], axis=0)
    print(
        f"{name:>5}: coefficient SD={np.round(coefficient_sd, 3)}, "
        f"mean test MSE={np.mean(test_errors[name]):.3f}"
    )
```

</details>

The first two variables carry almost the same information. OLS can move weight sharply from one to the other across bootstrap samples while preserving a similar combined prediction. Ridge makes the individual coefficients less variable and often improves prediction, but its shrunken coefficients no longer have the same unbiased-estimator interpretation as unpenalized OLS.

#### **Lasso Regression**

Lasso uses the $L_1$ norm:

$$
\widehat\beta_{\text{lasso}}
=\arg\min_\beta
\left\{
\frac{1}{2n}\lVert y-X\beta\rVert_2^2
+\lambda\lVert\beta\rVert_1
\right\}.
$$

The absolute-value penalty has corners at the coordinate axes. As the loss contours expand to touch the feasible region, the optimum often occurs at a corner, making some coefficients exactly zero. This creates an embedded sparse feature selector and can improve interpretability when the true signal is sparse.

Sparsity is not certainty. With strongly correlated predictors, lasso may select one almost arbitrarily and discard another that is equally plausible; small data changes can alter the selected support. Standard errors from ordinary least squares are also invalid after data-dependent selection. Stability checks, nested validation, and domain reasoning are needed before treating a selected variable as substantively important.

<details>
<summary><strong>Python example: recover a sparse signal with cross-validated lasso</strong></summary>

```python
import numpy as np
from sklearn.linear_model import Lasso
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(17)
X = rng.normal(size=(220, 20))
true_beta = np.zeros(20)
true_beta[[1, 6, 14]] = [2.5, -1.8, 1.2]
y = X @ true_beta + rng.normal(scale=1.0, size=len(X))

pipeline = Pipeline([
    ("scale", StandardScaler()),
    ("lasso", Lasso(max_iter=20_000)),
])
search = GridSearchCV(
    pipeline,
    {"lasso__alpha": np.logspace(-3, 0.5, 35)},
    scoring="neg_mean_squared_error",
    cv=KFold(5, shuffle=True, random_state=17),
)
search.fit(X, y)

coef = search.best_estimator_.named_steps["lasso"].coef_
selected = np.flatnonzero(np.abs(coef) > 1e-8)
print("true nonzero features:", np.flatnonzero(true_beta).tolist())
print("selected features:", selected.tolist())
print("selected standardized coefficients:", np.round(coef[selected], 3))
print("selected alpha:", round(search.best_params_["lasso__alpha"], 4))
```

</details>

#### **Elastic Net**

Elastic Net combines the $L_1$ and squared $L_2$ penalties:

$$
\widehat\beta
=\arg\min_\beta
\left\{
\frac{1}{2n}\lVert y-X\beta\rVert_2^2
+\lambda\left[
\rho\lVert\beta\rVert_1
+\frac{1-\rho}{2}\lVert\beta\rVert_2^2
\right]
\right\},
$$

where $\rho=1$ gives lasso and $\rho=0$ approaches ridge. The $L_1$ part permits exact zeros; the $L_2$ part encourages correlated predictors to enter or leave more as a group. Both $\lambda$ and the mixing ratio $\rho$ should be chosen by validation.

![Lasso and Elastic Net regularization paths show when coefficients enter the model as the penalty weakens.](assets/lasso-elasticnet-path.png){fig-align="center" width="82%" fig-alt="Coefficient paths for lasso and elastic net over regularization strengths"}

*Image: [scikit-learn, Lasso and Elastic Net Paths](https://scikit-learn.org/stable/auto_examples/linear_model/plot_lasso_lasso_lars_elasticnet_path.html).*

<details>
<summary><strong>Python example: compare lasso and Elastic Net with correlated signal features</strong></summary>

```python
import numpy as np
from sklearn.linear_model import ElasticNet, Lasso
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(9)
n = 180
shared = rng.normal(size=n)
X = np.column_stack([
    shared + rng.normal(scale=0.08, size=n),
    shared + rng.normal(scale=0.08, size=n),
    shared + rng.normal(scale=0.08, size=n),
    rng.normal(size=(n, 7)),
])
y = 2.0 * shared + rng.normal(scale=0.8, size=n)
cv = KFold(5, shuffle=True, random_state=9)

lasso_search = GridSearchCV(
    Pipeline([("scale", StandardScaler()), ("model", Lasso(max_iter=20_000))]),
    {"model__alpha": np.logspace(-3, 0, 25)},
    scoring="neg_mean_squared_error",
    cv=cv,
).fit(X, y)

elastic_search = GridSearchCV(
    Pipeline([("scale", StandardScaler()), ("model", ElasticNet(max_iter=20_000))]),
    {
        "model__alpha": np.logspace(-3, 0, 20),
        "model__l1_ratio": [0.1, 0.5, 0.9],
    },
    scoring="neg_mean_squared_error",
    cv=cv,
).fit(X, y)

for name, search in [("Lasso", lasso_search), ("Elastic Net", elastic_search)]:
    coef = search.best_estimator_.named_steps["model"].coef_
    print(
        f"{name:>11}: first three coefficients={np.round(coef[:3], 3)}, "
        f"nonzero count={np.count_nonzero(np.abs(coef) > 1e-8)}"
    )
```

</details>

| Method | Penalty | Exact zeros? | Typical reason to use it | Main caution |
|---|---:|:---:|---|---|
| Ridge | $\lVert\beta\rVert_2^2$ | No | Many weak or correlated predictors | Retains every feature |
| Lasso | $\lVert\beta\rVert_1$ | Yes | Sparse prediction and compact models | Unstable selection among correlated features |
| Elastic Net | Mixed $L_1/L_2$ | Yes | Sparse modeling with correlated groups | Requires tuning two penalty controls |

Regularization improves a fitted model only relative to a specified prediction problem. Coefficients, feature selection, and optimal penalty strength can change when the target population, feature scaling, loss function, or data split changes.


### **Multiclass Linear Classification**

Binary logistic regression produces one log-odds score. A target with $K>2$ classes requires a strategy for organizing several scores into one prediction. The main linear approaches either decompose the task into binary problems or fit all classes jointly with a normalized softmax likelihood.

#### **One-vs-Rest and One-vs-One**

**One-vs-Rest (OvR)** trains $K$ binary classifiers. Classifier $k$ separates class $k$ from the union of all other classes, and prediction selects the largest decision score. It needs only $K$ models and is easy to parallelize. However, each negative class is heterogeneous, and independently fitted binary probabilities do not naturally form one coherent distribution whose values sum to one.

**One-vs-One (OvO)** trains a classifier for every class pair, requiring

$$
\binom{K}{2}=\frac{K(K-1)}{2}
$$

models. At inference, pairwise votes or calibrated pairwise scores determine the winner. Each learner sees a smaller and often simpler two-class problem, which can help algorithms such as support vector machines. The cost grows quadratically with class count, and converting pairwise decisions into globally consistent probabilities is nontrivial.

<details>
<summary><strong>Python example: compare OvR and OvO decision organization</strong></summary>

```python
import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.multiclass import OneVsOneClassifier, OneVsRestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = make_classification(
    n_samples=600,
    n_features=8,
    n_informative=6,
    n_redundant=0,
    n_classes=3,
    n_clusters_per_class=1,
    class_sep=1.3,
    random_state=14,
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=14
)

models = {
    "OvR": OneVsRestClassifier(
        make_pipeline(StandardScaler(), LogisticRegression(max_iter=2_000))
    ),
    "OvO": OneVsOneClassifier(
        make_pipeline(StandardScaler(), LogisticRegression(max_iter=2_000))
    ),
}

for name, model in models.items():
    model.fit(X_train, y_train)
    prediction = model.predict(X_test)
    scores = model.decision_function(X_test[:2])
    print(
        f"{name}: binary estimators={len(model.estimators_)}, "
        f"accuracy={accuracy_score(y_test, prediction):.3f}"
    )
    print(" first two class-level decision scores:\n", np.round(scores, 3))
```

</details>

OvR and OvO are problem decompositions rather than new base learners. The binary estimator can be logistic regression, a support vector machine, or another classifier. Their score scales and calibration behavior depend on that base estimator.

#### **Softmax Regression**

Softmax regression, also called multinomial logistic regression, fits all classes jointly. For class $k$,

$$
z_k(x)=\beta_{0k}+x^\top\beta_k,
\qquad
p(Y=k\mid x)=
\frac{\exp(z_k)}{\sum_{j=1}^{K}\exp(z_j)}.
$$

The denominator couples the classes, so the probabilities are nonnegative and sum to one. Parameters minimize multiclass cross-entropy,

$$
-\sum_{i=1}^{n}\sum_{k=1}^{K}
\mathbb 1(y_i=k)\log p(Y=k\mid x_i),
$$

usually with regularization. Adding the same constant to every class score leaves softmax probabilities unchanged, so implementations impose an implicit or explicit identifiability convention. For numerical stability, software subtracts the largest logit before exponentiation.

Between classes $a$ and $b$, the decision boundary satisfies $z_a(x)=z_b(x)$, or

$$
(\beta_a-\beta_b)^\top x+(\beta_{0a}-\beta_{0b})=0.
$$

Thus each pairwise boundary is linear in the represented feature space. Basis expansions can bend boundaries in the original input space, but the model remains linear in the expanded features.

![Joint multinomial logistic regression and one-vs-rest logistic regression can produce different linear decision regions.](assets/multiclass-logistic-boundaries.png){fig-align="center" width="82%" fig-alt="Multiclass decision boundaries from multinomial and one-vs-rest logistic regression"}

*Image: [scikit-learn, Multinomial and One-vs-Rest Logistic Regression](https://scikit-learn.org/stable/auto_examples/linear_model/plot_logistic_multinomial.html).*

<details>
<summary><strong>Python example: reconstruct softmax probabilities from fitted logits</strong></summary>

```python
import numpy as np
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression

X, y = load_iris(return_X_y=True)
model = LogisticRegression(C=100.0, max_iter=5_000).fit(X, y)

rows = X[[0, 70, 140]]
logits = rows @ model.coef_.T + model.intercept_

# Subtract the row maximum before exponentiation to avoid overflow.
stable_logits = logits - logits.max(axis=1, keepdims=True)
unnormalized = np.exp(stable_logits)
manual_probabilities = unnormalized / unnormalized.sum(axis=1, keepdims=True)

print("manual softmax probabilities:\n", np.round(manual_probabilities, 4))
print("model probabilities:\n", np.round(model.predict_proba(rows), 4))
print("row sums:", manual_probabilities.sum(axis=1))
print("largest absolute difference:", np.max(np.abs(
    manual_probabilities - model.predict_proba(rows)
)))
```

</details>

<details>
<summary><strong>Python example: compare coherent OvR and multinomial probabilities</strong></summary>

```python
import numpy as np
from sklearn.datasets import make_blobs
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = make_blobs(
    n_samples=450,
    centers=[(-2, 0), (1, 2), (2, -1)],
    cluster_std=[1.4, 1.2, 1.5],
    random_state=6,
)
query = np.array([[0.0, 0.0], [1.0, 0.5]])

ovr = OneVsRestClassifier(
    make_pipeline(StandardScaler(), LogisticRegression(max_iter=2_000))
).fit(X, y)
softmax = make_pipeline(
    StandardScaler(), LogisticRegression(max_iter=2_000)
).fit(X, y)

print("OvR normalized probabilities:\n", np.round(ovr.predict_proba(query), 3))
print("joint softmax probabilities:\n", np.round(softmax.predict_proba(query), 3))
```

</details>

**Strategy comparison.** OvR is computationally economical and supports any binary classifier. OvO focuses on pairwise distinctions and can work well when each pair is easy to separate. Softmax is the natural joint probabilistic model when classes are mutually exclusive and calibrated class probabilities matter. These strategies do not solve class imbalance, label noise, or nonlinear structure by themselves; those issues still require suitable loss weighting, representations, validation metrics, and diagnostics.


### **Assumptions and Diagnostics**

A fitted coefficient table is not the end of a linear-model analysis. **Diagnostics** ask whether the model's functional form, variance, dependence, and influential observations are compatible with the intended use. They do not prove that a model is true; they expose recognizable ways in which conclusions can fail.

![Residual plots reveal different failures: random scatter is compatible with the mean model, curvature suggests misspecification, a funnel suggests heteroscedasticity, and isolated points may be influential.](assets/residual-diagnostic-patterns.svg){fig-align="center" width="100%" fig-alt="Four residual versus fitted plots showing healthy scatter, curvature, funnel variance, and an influential point"}

Diagnostics must follow the data-generating structure. Random train-test splitting, ordinary standard errors, and pointwise residual assumptions are inappropriate when observations are clustered, repeated, spatially dependent, or ordered in time.

#### **Linearity, Independence, and Homoscedasticity**

For OLS, the most important condition for unbiased conditional-mean coefficients is

$$
\mathbb E[\varepsilon\mid X]=0.
$$

It fails when omitted variables, selection, simultaneity, or measurement processes make the unexplained component systematically related to included predictors. A low residual mean in the training sample does not test this condition: an intercept forces sample residuals to sum to zero.

The familiar assumptions play different roles:

- **Linearity of the conditional mean:** $\mathbb E[Y\mid X]$ is additive and linear in the represented features. For a GLM, linearity is required on the link scale, $g(\mu)=X\beta$.
- **Independence:** observations do not share unmodeled dependence. Time series, repeated measures, households, classrooms, and geographic clusters commonly violate it.
- **Homoscedasticity:** $\operatorname{Var}(\varepsilon\mid X)=\sigma^2$ is constant. Heteroscedasticity does not automatically bias OLS coefficients when the conditional mean is correct, but conventional standard errors become wrong and OLS is no longer the most efficient linear unbiased estimator.
- **Approximate normality:** normal errors justify exact small-sample $t$ and $F$ inference. It is not required for computing OLS predictions, and large-sample inference often relies on asymptotic approximations instead.

For logistic regression, errors are not Gaussian and constant variance is not expected: $\operatorname{Var}(Y\mid X)=p(1-p)$. Relevant checks include independence, a correctly specified logit-scale relationship, adequate overlap, absence of complete separation, and calibration.

<details>
<summary><strong>Python example: detect heteroscedasticity and compare conventional with HC3 standard errors</strong></summary>

```python
import numpy as np
from scipy.stats import chi2

rng = np.random.default_rng(42)
n = 500
x = rng.uniform(0, 5, n)

# The conditional mean is linear, but noise grows with x.
noise_scale = 0.3 + 0.55 * x
y = 1.0 + 2.2 * x + rng.normal(scale=noise_scale)
X = np.column_stack([np.ones(n), x])

beta = np.linalg.lstsq(X, y, rcond=None)[0]
residual = y - X @ beta
xtx_inverse = np.linalg.inv(X.T @ X)
leverage = np.einsum("ij,jk,ik->i", X, xtx_inverse, X)

# Conventional covariance assumes one common residual variance.
sigma2 = residual @ residual / (n - X.shape[1])
conventional_covariance = sigma2 * xtx_inverse

# HC3 inflates each squared residual according to its leverage.
hc3_weight = (residual / (1 - leverage)) ** 2
hc3_meat = X.T @ (X * hc3_weight[:, None])
hc3_covariance = xtx_inverse @ hc3_meat @ xtx_inverse

# Breusch-Pagan LM test: regress squared residuals on the predictors.
aux_beta = np.linalg.lstsq(X, residual**2, rcond=None)[0]
aux_fitted = X @ aux_beta
aux_r2 = 1 - np.sum((residual**2 - aux_fitted) ** 2) / np.sum(
    (residual**2 - np.mean(residual**2)) ** 2
)
lm_stat = n * aux_r2
lm_pvalue = chi2.sf(lm_stat, df=X.shape[1] - 1)

print("slope estimate:", round(beta[1], 3))
print("conventional slope SE:", round(np.sqrt(conventional_covariance[1, 1]), 3))
print("HC3 robust slope SE:", round(np.sqrt(hc3_covariance[1, 1]), 3))
print("Breusch-Pagan p-value:", f"{lm_pvalue:.3g}")
```

</details>

Robust standard errors repair one inferential consequence of heteroscedasticity; they do not repair a wrong mean function, dependence, omitted confounding, or poor extrapolation. If variance is scientifically meaningful, a variance model, transformation, weighted least squares, or appropriate GLM may be preferable.

#### **Multicollinearity and Influential Observations**

**Multicollinearity** means one predictor is approximately reconstructable from others. It does not necessarily damage predictions inside the observed feature distribution, but it makes individual coefficient estimates sensitive because the data contain little information for allocating a shared effect.

For predictor $X_j$, the variance inflation factor is

$$
\operatorname{VIF}_j=\frac{1}{1-R_j^2},
$$

where $R_j^2$ comes from regressing $X_j$ on the remaining predictors. Large VIF values signal inflated coefficient variance, not a universal requirement to delete a variable. The design-matrix condition number provides a complementary global measure and is scale-dependent, so predictors should be standardized before interpreting it.

An unusual observation can matter in two different ways:

- **Leverage** $h_{ii}$, the $i$th diagonal of $H=X(X^\top X)^{-1}X^\top$, measures how unusual its predictor values are.
- **Residual size** measures disagreement between its observed and fitted response.

An observation becomes strongly **influential** when removing it materially changes the fit. Cook's distance combines leverage and residual information. A high-leverage point that follows the trend can have a small residual; a large residual near the feature center may have limited leverage. Neither should be deleted automatically: first verify data quality, understand the population represented, and report sensitivity with and without defensible cases.

![An influence plot combines studentized residuals, leverage, and point size proportional to Cook's distance.](assets/regression-influence-plot.png){fig-align="center" width="76%" fig-alt="Regression influence plot showing studentized residuals against leverage with bubble size for Cook distance"}

*Image: [statsmodels, Regression Diagnostics](https://www.statsmodels.org/stable/examples/notebooks/generated/regression_diagnostics.html).*

<details>
<summary><strong>Python example: compute VIF, leverage, and Cook's distance</strong></summary>

```python
import numpy as np

rng = np.random.default_rng(5)
n = 120
x1 = rng.normal(size=n)
x2 = 0.97 * x1 + rng.normal(scale=0.12, size=n)  # strong collinearity
x3 = rng.normal(size=n)
y = 1.0 + 1.5 * x1 + 1.5 * x2 - 0.7 * x3 + rng.normal(scale=0.7, size=n)

# Add one unusual predictor combination with an inconsistent response.
x1 = np.append(x1, 4.5)
x2 = np.append(x2, 4.4)
x3 = np.append(x3, -3.0)
y = np.append(y, -4.0)

features = np.column_stack([x1, x2, x3])
X = np.column_stack([np.ones(len(features)), features])
names = ["x1", "x2", "x3"]

# VIF uses an auxiliary regression of each feature on all other features.
vifs = {}
for column, name in enumerate(names):
    target = features[:, column]
    others = np.delete(features, column, axis=1)
    auxiliary_X = np.column_stack([np.ones(len(others)), others])
    fitted = auxiliary_X @ np.linalg.lstsq(auxiliary_X, target, rcond=None)[0]
    r2 = 1 - np.sum((target - fitted) ** 2) / np.sum((target - target.mean()) ** 2)
    vifs[name] = 1 / (1 - r2)

beta = np.linalg.lstsq(X, y, rcond=None)[0]
residual = y - X @ beta
xtx_inverse = np.linalg.inv(X.T @ X)
leverage = np.einsum("ij,jk,ik->i", X, xtx_inverse, X)
n_rows, p = X.shape
mse = residual @ residual / (n_rows - p)
studentized = residual / np.sqrt(mse * (1 - leverage))
cooks_d = (residual**2 / (p * mse)) * leverage / (1 - leverage) ** 2
top = np.argsort(cooks_d)[-3:][::-1]

print("VIF:", {name: round(float(value), 2) for name, value in vifs.items()})
for row in top:
    print(
        f"row={row:>3}, leverage={leverage[row]:.3f}, "
        f"studentized residual={studentized[row]:.2f}, "
        f"Cook D={cooks_d[row]:.3f}"
    )
```

</details>

#### **Residual Diagnostics**

An ordinary residual is $e_i=y_i-\widehat y_i$. Its scale and expected pattern depend on the fitted model, so no single plot answers every diagnostic question.

- **Residuals versus fitted values** reveal curvature, changing variance, and subgroups omitted from the mean model.
- **Residuals versus each predictor or time order** can expose local misspecification and dependence hidden in a fitted-value plot.
- **Normal Q-Q plots** assess whether standardized residual tails support Gaussian small-sample inference; they do not test linearity.
- **Scale-location plots** emphasize changes in residual spread.
- **Residuals versus leverage** help identify observations whose feature position and response discrepancy jointly affect the fit.

For GLMs, raw residuals have unequal variance by construction. **Pearson residuals** divide by the model-implied standard deviation; **deviance residuals** allocate each observation's contribution to model deviance. Binary models should additionally be checked with calibration curves, proper scoring rules, and performance across meaningful subgroups.

<details>
<summary><strong>Python example: use residual structure to detect an omitted nonlinear effect</strong></summary>

```python
import numpy as np
from scipy.stats import f

rng = np.random.default_rng(15)
x = rng.uniform(-2.5, 2.5, 260)
y = 1.0 + 1.4 * x + 1.1 * x**2 + rng.normal(scale=0.9, size=len(x))

linear_X = np.column_stack([np.ones(len(x)), x])
quadratic_X = np.column_stack([np.ones(len(x)), x, x**2])

def fit_ols(design, outcome):
    beta = np.linalg.lstsq(design, outcome, rcond=None)[0]
    residual = outcome - design @ beta
    return beta, residual, residual @ residual

_, linear_residual, linear_rss = fit_ols(linear_X, y)
_, quadratic_residual, quadratic_rss = fit_ols(quadratic_X, y)

# Nested-model F test asks whether the added x^2 term materially reduces RSS.
added_parameters = quadratic_X.shape[1] - linear_X.shape[1]
remaining_df = len(y) - quadratic_X.shape[1]
f_statistic = ((linear_rss - quadratic_rss) / added_parameters) / (
    quadratic_rss / remaining_df
)
nonlinearity_pvalue = f.sf(f_statistic, added_parameters, remaining_df)

# Average residuals in ordered x bins. A correct mean model should not leave
# a stable U-shaped pattern across these groups.
edges = np.quantile(x, np.linspace(0, 1, 7))
bin_id = np.clip(np.digitize(x, edges[1:-1]), 0, 5)
bin_means = [linear_residual[bin_id == group].mean() for group in range(6)]

# Constants common to both Gaussian AIC values are omitted; the difference is unchanged.
linear_aic = len(y) * np.log(linear_rss / len(y)) + 2 * linear_X.shape[1]
quadratic_aic = len(y) * np.log(quadratic_rss / len(y)) + 2 * quadratic_X.shape[1]

print("linear-model binned residual means:", np.round(bin_means, 2))
print("nested nonlinearity test p-value:", f"{nonlinearity_pvalue:.3g}")
print("linear AIC (up to a constant):", round(linear_aic, 1))
print("quadratic AIC (up to a constant):", round(quadratic_aic, 1))
```

</details>

A useful diagnostic cycle is: inspect data provenance and missingness; fit the prespecified model; examine residual structure and influence; trace each anomaly to a plausible mechanism; revise the representation, distribution, variance, or dependence model; and finally re-evaluate on untouched data. Repeatedly modifying a model until diagnostics look clean turns the diagnostics into another adaptive fitting step, so confirmatory claims require honest validation or new data.


### **Strengths, Limitations, and Model Choice**

Linear and generalized linear models remain strong default models because they make assumptions visible. They train quickly, optimize a convex objective in their standard forms, support sparse inputs, and provide a direct path from a fitted score to a prediction. They are also valuable diagnostic baselines: if a much more complex model cannot reliably outperform a regularized linear model, the additional complexity may not be justified.

Their interpretability is conditional rather than automatic. A coefficient has meaning only after specifying the feature units, reference groups, link scale, interactions, regularization, and conditioning variables. With correlated predictors, there may be many nearly equivalent coefficient allocations. With basis expansions, a single coefficient describes one basis function rather than the whole effect; partial-effect curves are more informative. Predictive coefficients should never be promoted to causal effects without a causal design.

| Modeling need | Suitable starting point | What it models | Important checks |
|---|---|---|---|
| Real-valued conditional mean | OLS or ridge | $\mathbb E[Y\mid X]$ with identity link | Residual mean structure, dependence, variance, influence |
| Binary event probability | Logistic regression | Log-odds and Bernoulli probability | Separation, calibration, class-specific performance |
| Mutually exclusive multiclass target | Softmax regression | Joint class probabilities | Calibration, imbalance, confusion structure |
| Counts or exposure-adjusted rates | Poisson GLM with offset | Log conditional rate | Overdispersion, excess zeros, exposure definition |
| Smooth nonlinear effect | Splines or a GAM | Additive effects on the link scale | Knot/smoothness tuning, interactions, boundary behavior |
| Many correlated predictors | Ridge | Dense, stabilized coefficient vector | Scaling and validation of $\lambda$ |
| Sparse predictive signal | Lasso | Selected coefficient support | Selection stability and post-selection interpretation |
| Sparse correlated groups | Elastic Net | Mixed sparse and grouped shrinkage | Joint tuning of penalty strength and mixture |

The main limitations follow from the same structure that creates transparency:

- additive predictors cannot express an interaction unless it is represented explicitly;
- linear or smooth link-scale effects may miss thresholds, discontinuities, and complex local structure;
- squared-error regression can be sensitive to outcome outliers;
- ordinary formulas fail under dependence, heteroscedasticity, selection, or distributional misspecification;
- predictions can extrapolate to implausible values outside the observed feature range;
- sparse penalties can make feature selection look more certain than the data support.

A defensible model choice proceeds in the following order:

1. Define the prediction unit, target population, outcome support, and decision-relevant estimand.
2. Choose a split strategy that respects groups, time, and repeated entities.
3. Construct features using training-only pipelines; include interactions or smooth effects only when justified.
4. Match the response family and link to the outcome and its mean-variance relationship.
5. Treat regularization strength, basis complexity, and thresholds as hyperparameters selected on development data.
6. Use metrics aligned with the task: squared error for conditional means, log loss for probabilities, deviance for count models, and calibration when probabilities drive decisions.
7. Examine residuals, calibration, subgroup performance, coefficient stability, and influential observations.
8. Evaluate the final locked pipeline once on untouched test data, then monitor distribution shift after deployment.

<details>
<summary><strong>Python example: compare linear candidates on development data and test once</strong></summary>

```python
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.linear_model import ElasticNet, LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import SplineTransformer, StandardScaler

X, y = load_diabetes(return_X_y=True)
X_development, X_test, y_development, y_test = train_test_split(
    X, y, test_size=0.2, random_state=27
)
cv = KFold(n_splits=5, shuffle=True, random_state=27)

candidates = {
    "OLS": (
        Pipeline([("scale", StandardScaler()), ("model", LinearRegression())]),
        {},
    ),
    "Ridge": (
        Pipeline([("scale", StandardScaler()), ("model", Ridge())]),
        {"model__alpha": np.logspace(-3, 3, 13)},
    ),
    "Elastic Net": (
        Pipeline([
            ("scale", StandardScaler()),
            ("model", ElasticNet(max_iter=30_000)),
        ]),
        {
            "model__alpha": np.logspace(-3, 1, 10),
            "model__l1_ratio": [0.2, 0.5, 0.8],
        },
    ),
    "Spline + Ridge": (
        Pipeline([
            ("basis", SplineTransformer(degree=2, include_bias=False)),
            ("scale", StandardScaler()),
            ("model", Ridge()),
        ]),
        {
            "basis__n_knots": [3, 4, 5],
            "model__alpha": [0.1, 1.0, 10.0, 100.0],
        },
    ),
}

searches = {}
for name, (pipeline, grid) in candidates.items():
    search = GridSearchCV(
        pipeline,
        grid,
        scoring="neg_mean_squared_error",
        cv=cv,
    ).fit(X_development, y_development)
    searches[name] = search
    cv_rmse = np.sqrt(-search.best_score_)
    print(f"{name:>14}: development CV RMSE={cv_rmse:.2f}")

# Select using development CV only. The test set has not influenced this choice.
winner_name = max(searches, key=lambda name: searches[name].best_score_)
winner = searches[winner_name].best_estimator_
test_prediction = winner.predict(X_test)

print("selected model:", winner_name)
print("untouched test RMSE:", round(np.sqrt(mean_squared_error(y_test, test_prediction)), 2))
print("untouched test R2:", round(r2_score(y_test, test_prediction), 3))
```

</details>

The winning name in one split is not a universal ranking. It reflects this dataset, candidate set, metric, and validation design. When candidates are close relative to fold-to-fold uncertainty, prefer the simpler or operationally safer model and report the uncertainty instead of declaring a decisive winner.

Linear modeling is therefore best understood as a disciplined workflow rather than one formula: represent the effect, choose a response distribution and link, regularize at the scale supported by the data, validate the whole pipeline, and diagnose the assumptions that make predictions and interpretations trustworthy.
